# Atom typing with atom_bond.xml (ACT logic)

This notebook loads a PDB/XYZ/SDF file and uses the **exact same Python functions as ACT** (`MoleculeDict.read` and `lookUpSpecial`) to assign atom types via `atom_bond.xml`. \
It prints detailed SMARTS hits and assigned types, and renders an image of the molecule with labels.
Change in the config block to your test molecule and run all blocks.\
You can also use nother atom_bond.xml by copying it in the CWD of the notebook, `get_mol_dict.py` will prioritize it.


In [1]:
from pathlib import Path
import os, sys
import xmltodict
from rdkit import Chem
from rdkit.Chem import Draw, AllChem

from get_mol_dict import MoleculeDict, get_atom_bond_xml

os.environ['ACTDATA'] = str(Path('../..') / 'share')


In [2]:
# Config
input_path = Path("1-methyl-adenine.xyz") 
mycharge = 0  

In [3]:
def collect_hits(mol, abe):
    hits = []
    for entry in abe:
        pattern = Chem.MolFromSmarts(entry["smarts"])
        if pattern is None:
            continue
        matches = mol.GetSubstructMatches(pattern)
        if not matches:
            continue
        hits.append({
            "name": entry["name"],
            "smarts": entry["smarts"],
            "count": len(matches),
            "matches": matches,
        })
    return hits


In [4]:
#  Run using the exact ACT functions
abe = get_atom_bond_xml()

md = MoleculeDict()
success = md.read(input_path.stem, str(input_path), input_path.suffix.lstrip("."), mycharge)
if not success:
    raise RuntimeError("MoleculeDict.read() failed")

# Use the RDKit mol created by MoleculeDict for SMARTS hits and rendering
mol = getattr(md, "mol", None)
if mol is None:
    raise RuntimeError("MoleculeDict did not creat RDKit mol object")

print(f"Title: {md.title}")
print(f"Formula: {md.formula}")
print(f"Atoms: {len(md.atoms)}  Bonds: {len(md.bonds)}")


Title: 1-methyl-adenine
Formula: C6H7N5
Atoms: 18  Bonds: 19


In the following **ALL** SMART hits are printed and not only the first hit.

In [5]:
# SMARTS hits with match atom indices
hits = collect_hits(mol, abe)
print(f"SMARTS entries hit: {len(hits)}")
for h in hits:
    print(f"- {h['name']}  count={h['count']}  smarts={h['smarts']}")
    for m in h["matches"]:
        print("   match:", m)


SMARTS entries hit: 10
- gaff_h5_ring_2EWG  count=2  smarts=[#1X1;$(*-[c](~[#7,#8,#16,#17,#35,#53])~[#7,#8,#16,#17,#35,#53])]
   match: (13,)
   match: (17,)
- gaff_h4_ring_1EWG  count=2  smarts=[#1X1;$(*-[c]~[#7,#8,#16,#17,#35,#53])]
   match: (13,)
   match: (17,)
- gaff_h1_chain_1EWG  count=3  smarts=[#1X1;$(*-[C]-[#7,#8,#9,#16,#17,#35,#53])]
   match: (14,)
   match: (15,)
   match: (16,)
- gaff_hc_aliphatic_C  count=3  smarts=[#1X1;$(*-[#6X4])]
   match: (14,)
   match: (15,)
   match: (16,)
- gaff_hn_nitrogen  count=2  smarts=[#1X1;$(*-N)]
   match: (11,)
   match: (12,)
- gaff_ha_generic  count=7  smarts=[#1X1]
   match: (11,)
   match: (12,)
   match: (13,)
   match: (14,)
   match: (15,)
   match: (16,)
   match: (17,)
- gaff_hc_any  count=7  smarts=[#1]
   match: (11,)
   match: (12,)
   match: (13,)
   match: (14,)
   match: (15,)
   match: (16,)
   match: (17,)
- c_sp2  count=5  smarts=[#6;^2]
   match: (1,)
   match: (2,)
   match: (3,)
   match: (5,)
   match: (9,)
- c_sp

Here the assigned atoms and bonds through get_mol_dict.py are displayed.

In [6]:
# Assigned atom types
print(f"{'idx':>3}  {'el':>2}  {'obtype':<8}")
for i, atom in enumerate(mol.GetAtoms()):
    print(f"{i:3d}  {atom.GetSymbol():2s}  {md.atoms[i]['obtype']:<8}")


idx  el  obtype  
  0  N   n2      
  1  C   c2      
  2  C   c2      
  3  C   c2      
  4  N   n2      
  5  C   c2      
  6  N   n2      
  7  C   c3      
  8  N   n2      
  9  C   c2      
 10  N   n2      
 11  H   hn      
 12  H   hn      
 13  H   h5      
 14  H   h1      
 15  H   h1      
 16  H   h1      
 17  H   h5      


In [7]:
# Bonds
print(f"{'ai':>3}  {'aj':>3}  {'order':>5}")
for (ai, aj), order in sorted(md.bonds.items()):
    print(f"{ai-1:3d}  {aj-1:3d}  {order:5.1f}")


 ai   aj  order
  1    0    1.0
  2    1    1.5
  3    2    1.0
  4    3    1.5
  5    4    1.5
  6    1    1.5
  6    5    1.5
  7    6    1.0
  8    3    1.5
  9    8    1.5
 10    2    1.5
 10    9    1.5
 11    0    1.0
 12    0    1.0
 13    5    1.0
 14    7    1.0
 15    7    1.0
 16    7    1.0
 17    9    1.0


In [9]:
# Draw molecule in 3D (py3Dmol) using existing coordinates + atomtype labels
import py3Dmol

# some variables
show_bond_labels = True
label_font = 12
bond_label_font = 10

# Use coordinates from the input file 
conf = mol.GetConformer()

molblock = Chem.MolToMolBlock(mol)
view = py3Dmol.view(width=800, height=500)
view.setBackgroundColor('#111111')
view.addModel(molblock, 'mol')
view.setStyle({'stick': {}})

# better colors
view.setStyle({'elem': 'C'}, {'stick': {'color': '#808080'}})
view.setStyle({'elem': 'H'}, {'stick': {'color': '#FFFFFF'}})
view.setStyle({'elem': 'N'}, {'stick': {'color': '#2B6CB0'}})
view.setStyle({'elem': 'O'}, {'stick': {'color': "#B82823"}})

# add atom-type labels at atomic positions
for i, atom in enumerate(mol.GetAtoms()):
    atype = md.atoms[i]['obtype']
    pos = conf.GetAtomPosition(i)
    view.addLabel(
        f"{i}:{atype}",
        {
            'position': {'x': pos.x, 'y': pos.y, 'z': pos.z},
            'fontColor': 'black',
            'fontSize': label_font,
            'showBackground': True,
            'backgroundColor': 'white',
            'backgroundOpacity': 0.6,
            'borderColor': 'black',
            'inFront': True,
        },
    )

# Add bond labels at bond midpoints
if show_bond_labels:
    for bond in mol.GetBonds():
        ai = bond.GetBeginAtomIdx()
        aj = bond.GetEndAtomIdx()
        posi = conf.GetAtomPosition(ai)
        posj = conf.GetAtomPosition(aj)
        mx = (posi.x + posj.x) / 2.0
        my = (posi.y + posj.y) / 2.0
        mz = (posi.z + posj.z) / 2.0
        order = md.bonds.get((ai + 1, aj + 1)) or md.bonds.get((aj + 1, ai + 1))
        if order is None:
            order = bond.GetBondTypeAsDouble()
        view.addLabel(
            f"{order:g}",
            {
                'position': {'x': mx, 'y': my, 'z': mz},
                'fontColor': 'black',
                'fontSize': bond_label_font,
                'showBackground': True,
                'backgroundColor': 'white',
                'backgroundOpacity': 0.6,
                'borderColor': 'black',
                'inFront': True,
            },
        )

#view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.